# 10 — One State, Five Compatible Views

Qubit states appear in many forms across different tools and textbooks:

- **Spinor** — a pair of complex amplitudes $(\alpha, \beta)$
- **Bloch vector** — a point $(x, y, z)$ on or inside the unit sphere
- **Quaternion** — a four-component rotation descriptor $(w, x, y, z)$
- **SU(2) matrix** — a 2×2 unitary that acts on state space
- **Qiskit circuit** — an executable workflow with a simulator result

These are not five different things. They are **one state seen through five compatible views**.

This notebook takes a single quantum state and walks it through every layer of the RQM stack. By the end you will see the whole system in one place.

---

## The canonical state

We will use the **|+⟩** state throughout:

$$|{+}\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$$

This is one of the cleanest non-trivial single-qubit states: it is normalized, has equal probability of measuring 0 or 1, and its Bloch vector points along the +X axis — making every layer easy to verify by inspection.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import (
    draw_bloch_sphere,
    plot_quaternion_components,
    plot_counts,
)
from helpers.display_utils import (
    show_state_vector,
    show_matrix,
    show_info_table,
    format_complex_pair,
)
setup_notebook()
print("Setup complete.")


---

## View 1 — Spinor

A single-qubit state is a **spinor**: a unit vector in $\mathbb{C}^2$.

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1$$

For $|{+}\rangle$:
$$\alpha = \frac{1}{\sqrt{2}}, \quad \beta = \frac{1}{\sqrt{2}}$$


In [ ]:
# Define |+> as a spinor
alpha = 1 / np.sqrt(2)
beta  = 1 / np.sqrt(2)

state = np.array([alpha, beta], dtype=complex)

print("Spinor representation of |+>:")
print(f"  {format_complex_pair(alpha, beta)}")
print()
print(f"  |alpha|^2 = {abs(alpha)**2:.4f}  (prob of |0>)")
print(f"  |beta|^2  = {abs(beta)**2:.4f}  (prob of |1>)")
print(f"  norm      = {np.linalg.norm(state):.6f}  (should be 1)")

print()
print("State vector as column:")
show_state_vector(state, basis_labels=[r"|0\rangle", r"|1\rangle"], label=r"|{+}\rangle")


---

## View 2 — Bloch vector

The Bloch vector maps the spinor to a point $(x, y, z)$ on the unit sphere.

For a state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$, the Bloch components are:

$$x = 2\,\text{Re}(\alpha^*\beta), \qquad y = 2\,\text{Im}(\alpha^*\beta), \qquad z = |\alpha|^2 - |\beta|^2$$

For $|{+}\rangle$, this gives $(x, y, z) = (1, 0, 0)$ — the +X pole of the Bloch sphere.

> **Global phase invariance:** multiplying $|\psi\rangle$ by $e^{i\phi}$ does not change the Bloch vector, because it cancels in the products above. The Bloch sphere removes the unobservable global phase.


In [ ]:
# Compute Bloch vector from spinor
bx = 2 * np.real(np.conj(alpha) * beta)
by = 2 * np.imag(np.conj(alpha) * beta)
bz = abs(alpha)**2 - abs(beta)**2
bloch = np.array([bx, by, bz])

print(f"Bloch vector of |+>:")
print(f"  x = {bx:.4f}  (2 Re(alpha* beta))")
print(f"  y = {by:.4f}  (2 Im(alpha* beta))")
print(f"  z = {bz:.4f}  (|alpha|^2 - |beta|^2)")
print(f"  |bloch| = {np.linalg.norm(bloch):.6f}  (unit vector for pure state)")

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
draw_bloch_sphere(
    ax=ax,
    vectors=[(bloch, r"$|{+}\rangle$", "steelblue")],
    title="Bloch sphere — |+⟩ at the +X pole",
)
plt.tight_layout()
plt.show()


---

## View 3 — Quaternion

The Bloch vector $(x, y, z)$ corresponds to a rotation axis.  
To rotate $|0\rangle$ (the +Z pole) to $|{+}\rangle$ (the +X pole), we rotate by **90° around the Y axis**.

The unit quaternion for this rotation is:
$$q = \cos\!\tfrac{\theta}{2} + \sin\!\tfrac{\theta}{2}\,(n_x\mathbf{i} + n_y\mathbf{j} + n_z\mathbf{k})$$

With $\theta = 90°$ and axis $(0, 1, 0)$:
$$q = \cos 45° + \sin 45°\,\mathbf{j} = \tfrac{1}{\sqrt{2}} + \tfrac{1}{\sqrt{2}}\,\mathbf{j}$$


In [ ]:
# Quaternion for Ry(90°): rotate |0> to |+>
theta = np.pi / 2  # 90 degrees
axis  = np.array([0.0, 1.0, 0.0])  # Y axis

w = np.cos(theta / 2)
xyz = np.sin(theta / 2) * axis
q = (w, xyz[0], xyz[1], xyz[2])

print("Quaternion for Ry(90°) — rotating |0⟩ to |+⟩:")
show_info_table([
    ("w (scalar)",       f"{q[0]:.6f}"),
    ("x",                f"{q[1]:.6f}"),
    ("y",                f"{q[2]:.6f}"),
    ("z",                f"{q[3]:.6f}"),
    ("|q|",              f"{np.sqrt(sum(v**2 for v in q)):.6f}  (unit quaternion)"),
    ("axis",             f"({axis[0]:.2f}, {axis[1]:.2f}, {axis[2]:.2f})  = Y"),
    ("angle",            f"{np.degrees(theta):.1f}°"),
], title="Quaternion — Ry(90°)")

fig = plot_quaternion_components(
    [q],
    labels=["Ry(90°)"],
    title="Quaternion components — rotation that maps |0⟩ to |+⟩",
)
plt.tight_layout()
plt.show()


---

## View 4 — SU(2) matrix

Every unit quaternion $(w, x, y, z)$ corresponds to an SU(2) matrix:

$$U = \begin{pmatrix} w + iz & y + ix \\ -y + ix & w - iz \end{pmatrix}$$

For $q = (\tfrac{1}{\sqrt{2}}, 0, \tfrac{1}{\sqrt{2}}, 0)$ (Ry(90°)):

$$R_y(90°) = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -1 \\ 1 & \phantom{-}1 \end{pmatrix}$$

Applying this to $|0\rangle = (1, 0)^T$ gives $|{+}\rangle = \tfrac{1}{\sqrt{2}}(1, 1)^T$, confirming all views are consistent.


In [ ]:
# SU(2) matrix from quaternion (w, x, y, z)
qw, qx, qy, qz = q
U = np.array([
    [qw + 1j*qz,   qy + 1j*qx],
    [-qy + 1j*qx,  qw - 1j*qz],
], dtype=complex)

print("SU(2) matrix for Ry(90°):")
show_matrix(U, label=r"R_y(90°)")

# Verify: apply to |0>
ket_0 = np.array([1.0, 0.0], dtype=complex)
result = U @ ket_0
print()
print("Applying Ry(90°) to |0⟩:")
show_state_vector(result, basis_labels=[r"|0\rangle", r"|1\rangle"])
print(f"  = ({result[0]:.4f}, {result[1]:.4f})")
print(f"  Expected: ({1/np.sqrt(2):.4f}, {1/np.sqrt(2):.4f})  ✓")

# Verify unitarity
print()
print("Unitarity check: U† U = I")
Udag_U = U.conj().T @ U
show_matrix(Udag_U, label=r"U^\dagger U")


---

## View 5 — Qiskit circuit and simulator

The Ry(90°) rotation is directly expressible as a Qiskit gate.  
Running the circuit on a simulator verifies that measuring the resulting state gives 50/50 outcomes — exactly what we expect for $|{+}\rangle$.


In [ ]:
try:
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator

    # Build the circuit: Ry(90°) on |0>, then measure
    qc = QuantumCircuit(1, 1)
    qc.ry(np.pi / 2, 0)   # Ry(90°)
    qc.measure(0, 0)

    print("Circuit:")
    print(qc.draw(output="text"))

    # Run on simulator
    sim = AerSimulator()
    job = sim.run(qc, shots=1000, seed_simulator=42)
    counts = job.result().get_counts()

    print(f"\nSimulator results (1000 shots): {counts}")
    print("Expected: approximately 50% |0⟩ and 50% |1⟩")

    fig, ax = plt.subplots(figsize=(4, 3))
    plot_counts(counts, title="Simulator — Ry(90°)|0⟩ measurement", ax=ax)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("qiskit or qiskit-aer not installed — skipping circuit execution.")
    print("Expected result: ~500 counts for '0' and ~500 counts for '1'.")
    # Show a representative example anyway
    counts = {"0": 498, "1": 502}
    fig, ax = plt.subplots(figsize=(4, 3))
    plot_counts(counts, title="Expected: Ry(90°)|0⟩ measurement (example)", ax=ax)
    plt.tight_layout()
    plt.show()


---

## Synthesis — All five views

Here is the complete picture for the single state $|{+}\rangle$:

| View | Object | Meaning |
|---|---|---|
| **Spinor** | $(\alpha, \beta) = (\tfrac{1}{\sqrt{2}}, \tfrac{1}{\sqrt{2}})$ | State amplitudes; squared magnitudes give measurement probabilities |
| **Bloch** | $(x, y, z) = (1, 0, 0)$ | Geometric orientation on the unit sphere — +X pole |
| **Quaternion** | $(w, x, y, z) = (\tfrac{1}{\sqrt{2}}, 0, \tfrac{1}{\sqrt{2}}, 0)$ | Rotation that takes \|0⟩ to \|+⟩ — 90° around Y |
| **SU(2)** | $R_y(90°) = \tfrac{1}{\sqrt{2}}\begin{pmatrix}1&-1\\1&1\end{pmatrix}$ | Group element acting on state space |
| **Qiskit** | `qc.ry(π/2, 0)` + simulator | Executable workflow verifying 50/50 measurement statistics |

These are not five disconnected math languages.  
This is **one state flowing through one coherent stack**.

- `rqm-core` is not abstract for its own sake.
- `rqm-qiskit` is not disconnected from the geometry.
- The notebook layer is not cosmetic.
- The whole stack is one system.


In [ ]:
# Visual summary: all five views together
fig = plt.figure(figsize=(14, 4))

# --- Bloch sphere ---
ax1 = fig.add_subplot(131, projection="3d")
draw_bloch_sphere(
    ax=ax1,
    vectors=[(np.array([1, 0, 0]), r"$|{+}\rangle$", "steelblue")],
    title="Bloch vector",
)

# --- Quaternion bar chart ---
ax2 = fig.add_subplot(132)
q_vals = list(q)
comp_names = ["w", "x", "y", "z"]
colours = ["#2563EB", "#7C3AED", "#059669", "#D97706"]
bars = ax2.bar(comp_names, q_vals, color=colours, alpha=0.85, edgecolor="white")
for bar, val in zip(bars, q_vals):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + (0.02 if val >= 0 else -0.08),
        f"{val:.3f}", ha="center", va="bottom", fontsize=9,
    )
ax2.axhline(0, color="black", linewidth=0.7)
ax2.set_ylim(-1.2, 1.2)
ax2.set_title("Quaternion components", fontsize=11)
ax2.spines[["top", "right"]].set_visible(False)

# --- Simulator counts ---
ax3 = fig.add_subplot(133)
plot_counts({"0": 498, "1": 502}, title="Simulator (50/50 expected)", ax=ax3)

plt.suptitle(r"$|{+}\rangle$ — one state, five compatible views", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## Where to go next

- **[00_welcome_and_repo_map.ipynb](00_welcome_and_repo_map.ipynb)** — return to the overview and choose a learning track
- **[01_quaternion_intuition.ipynb](01_quaternion_intuition.ipynb)** — go deeper on quaternion components
- **[02_spinor_to_bloch.ipynb](02_spinor_to_bloch.ipynb)** — go deeper on spinor ↔ Bloch connections
- **[05_rqm_qiskit_single_qubit_workflows.ipynb](05_rqm_qiskit_single_qubit_workflows.ipynb)** — build production-ready circuits
